# 03 -- ArcFace Multi-Dataset (v4: hard mining + extended training)

**v4 improvements over v3:**
- Hard example mining: after each epoch, classes with higher loss are oversampled
- Extended: Phase 1 = 30 epochs (was 20), Phase 2 = 15 epochs (was 10)
- LR warmup: 3-epoch linear warmup before cosine decay
- Gradient clipping (max_norm=1.0) for stable pretrained fine-tuning
- ArcFace per-sample loss returned to drive the sampler

**Training corpus**: 251 writers, 6884 genuine images
**Evaluation**: CEDAR test (9 writers) -- standard + hard-negative


---
## Section 0 -- Setup

In [ ]:
import sys, re, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn import metrics as sk_metrics
from sklearn.decomposition import PCA

def _find_project_root(start):
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'requirements.txt').exists():
            return p
    raise RuntimeError(f'Cannot find project root from {start}')

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.metrics.verification import compute_metrics

SEED    = 42
set_seed(SEED)
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DS_ROOT = PROJECT_ROOT / 'datasets'
print('Device      :', DEVICE)
print('Dataset root:', DS_ROOT)


---
## Section 1 -- Multi-Dataset Scanners

In [ ]:
def scan_cedar(root):
    root = Path(root)
    pat_o = re.compile(r'^original_(\d+)_(\d+)\.png$',  re.I)
    pat_f = re.compile(r'^forgeries_(\d+)_(\d+)\.png$', re.I)
    rows  = []
    for fp in (root/'full_org').iterdir():
        m = pat_o.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'cedar_{int(m.group(1)):03d}','label':'genuine','dataset':'cedar'})
    for fp in (root/'full_forg').iterdir():
        m = pat_f.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'cedar_{int(m.group(1)):03d}','label':'forgery','dataset':'cedar'})
    return pd.DataFrame(rows)

def scan_gpds150(root):
    root  = Path(root)
    pat_g = re.compile(r'^c-(\d+)-(\d+)',  re.I)
    pat_f = re.compile(r'^cf-(\d+)-(\d+)', re.I)
    rows  = []
    for fp in (root/'train'/'genuine').iterdir():
        m = pat_g.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'genuine','dataset':'gpds150'})
    for fp in (root/'train'/'forge').iterdir():
        m = pat_f.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'forgery','dataset':'gpds150'})
    for wd in (root/'test').iterdir():
        if not wd.is_dir(): continue
        for fp in (wd/'genuine').iterdir():
            m = pat_g.match(fp.name)
            if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'genuine','dataset':'gpds150'})
        for fp in (wd/'forge').iterdir():
            m = pat_f.match(fp.name)
            if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'forgery','dataset':'gpds150'})
    return pd.DataFrame(rows)

def scan_sigcomp2011(root):
    base = Path(root)/'trainingSet'/'OfflineSignatures'
    rows = []
    for lang, prefix in [('Dutch','sc11d'),('Chinese','sc11c')]:
        gen = base/lang/'TrainingSet'/'Offline Genuine'
        pat = re.compile(r'^(\d+)_', re.I)
        for fp in gen.iterdir():
            m = pat.match(fp.name)
            if m and fp.is_file():
                rows.append({'path':str(fp),'writer_uid':f'{prefix}_{int(m.group(1)):03d}','label':'genuine','dataset':f'sc2011_{lang.lower()}'})
    return pd.DataFrame(rows)

def scan_sigcomp2009(root):
    data = Path(root)/'NISDCC-offline-all-001-051-6g'/'NISDCC-offline-all-001-051-6g'
    pat  = re.compile(r'^NISDCC-(\d+)_', re.I)
    rows = []
    for fp in data.iterdir():
        m = pat.match(fp.name)
        if m and fp.is_file():
            rows.append({'path':str(fp),'writer_uid':f'sc09_{int(m.group(1)):03d}','label':'genuine','dataset':'sc2009'})
    return pd.DataFrame(rows)

print('Scanning...')
df_cedar = scan_cedar(DS_ROOT/'CEDAR')
df_gpds  = scan_gpds150(DS_ROOT/'GPDS150')
df_sc11  = scan_sigcomp2011(DS_ROOT/'sigComp2011-trainingSet')
df_sc09  = scan_sigcomp2009(DS_ROOT/'SigComp2009-training')
df_all   = pd.concat([df_cedar,df_gpds,df_sc11,df_sc09], ignore_index=True)
print(df_all.groupby(['dataset','label'])['writer_uid'].agg(writers='nunique',images='count').to_string())
print(f'\nTotal unique writers: {df_all["writer_uid"].nunique()}  |  Total images: {len(df_all)}')


---
## Section 2 -- Training / Evaluation Split

In [ ]:
cedar_writers = np.array(sorted(df_cedar['writer_uid'].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(cedar_writers)
n = len(cedar_writers); n_tr = int(round(n*0.70)); n_v = int(round(n*0.15))
cedar_train_wids = set(cedar_writers[:n_tr])
cedar_val_wids   = set(cedar_writers[n_tr:n_tr+n_v])
cedar_test_wids  = set(cedar_writers[n_tr+n_v:])

other_train_wids = (set(df_gpds['writer_uid'].unique()) |
                    set(df_sc11['writer_uid'].unique()) |
                    set(df_sc09['writer_uid'].unique()))
all_train_wids  = cedar_train_wids | other_train_wids
sorted_train    = sorted(all_train_wids)
writer_to_class = {wid: i for i, wid in enumerate(sorted_train)}
NUM_CLASSES     = len(sorted_train)

print(f'CEDAR split -- train:{len(cedar_train_wids)}  val:{len(cedar_val_wids)}  test:{len(cedar_test_wids)}')
print(f'Total training classes: {NUM_CLASSES}')


---
## Section 3 -- Transforms and Datasets

In [ ]:
IMG_SIZE  = 224
BATCH     = 64

train_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=8, translate=(0.05,0.05), scale=(0.93,1.07), fill=255),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
    T.Lambda(lambda x: x + 0.015*torch.randn_like(x)),
])
eval_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])

class SignatureClassDataset(Dataset):
    def __init__(self, df, writer_to_class, writer_uid_set, transform=None):
        sub = df[(df['writer_uid'].isin(writer_uid_set)) & (df['label']=='genuine')]
        self.records   = sub[['path','writer_uid']].reset_index(drop=True)
        self.w2c       = writer_to_class
        self.transform = transform
    def __len__(self): return len(self.records)
    def get_class_indices(self):
        return torch.tensor([self.w2c[uid] for uid in self.records['writer_uid']])
    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row['path']).convert('L')
        if self.transform: img = self.transform(img)
        return img, self.w2c[row['writer_uid']]

class SiamesePairDataset(Dataset):
    def __init__(self, pairs_df, transform=None):
        self.pairs = pairs_df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]
        a = Image.open(row['path_a']).convert('L')
        b = Image.open(row['path_b']).convert('L')
        if self.transform: a = self.transform(a); b = self.transform(b)
        return a, b, torch.tensor(row['label'], dtype=torch.float32)

train_cls_ds  = SignatureClassDataset(df_all, writer_to_class, all_train_wids, transform=train_tfm)
class_idx_all = train_cls_ds.get_class_indices()  # pre-computed once

def make_mining_loader(class_weights):
    """DataLoader where each sample's weight = its class difficulty."""
    sample_w = class_weights[class_idx_all].clamp(min=1e-6)
    sampler  = WeightedRandomSampler(sample_w.tolist(), len(sample_w), replacement=True)
    return DataLoader(train_cls_ds, batch_size=BATCH, sampler=sampler,
                      num_workers=0, pin_memory=DEVICE.type=='cuda')

x_sample, y_sample = next(iter(make_mining_loader(torch.ones(NUM_CLASSES))))
print(f'Train images  : {len(train_cls_ds):,}  across {NUM_CLASSES} classes')
print(f'Batch shape   : {x_sample.shape}')


---
## Section 4 -- ArcFace Loss

Added `reduction` parameter so the training loop can extract per-sample losses for hard mining.
With 251 classes, standard scale=64 and margin=0.5 are appropriate.


In [ ]:
class ArcFaceLoss(nn.Module):
    def __init__(self, emb_dim, num_classes, scale=64.0, margin=0.5):
        super().__init__()
        self.scale  = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.empty(num_classes, emb_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th    = math.cos(math.pi - margin)
        self.mm    = math.sin(math.pi - margin) * margin
    def forward(self, embeddings, labels, reduction='mean'):
        emb_n     = F.normalize(embeddings, dim=1)
        w_n       = F.normalize(self.weight, dim=1)
        cos_t     = F.linear(emb_n, w_n).clamp(-1+1e-7, 1-1e-7)
        sin_t     = torch.sqrt(1.0 - cos_t**2)
        cos_tm    = cos_t * self.cos_m - sin_t * self.sin_m
        cos_tm    = torch.where(cos_t > self.th, cos_tm, cos_t - self.mm)
        one_hot   = torch.zeros_like(cos_t).scatter_(1, labels.view(-1,1).long(), 1.0)
        output    = ((one_hot * cos_tm) + ((1-one_hot) * cos_t)) * self.scale
        return F.cross_entropy(output, labels.long(), reduction=reduction)

EMB_DIM = 256
_arc = ArcFaceLoss(EMB_DIM, NUM_CLASSES).to(DEVICE)
_e   = torch.randn(4, EMB_DIM).to(DEVICE)
_l   = torch.randint(0, NUM_CLASSES, (4,)).to(DEVICE)
print('ArcFace mean loss :', _arc(_e, _l, reduction='mean').item())
print('ArcFace per-sample:', _arc(_e, _l, reduction='none').tolist())
del _arc, _e, _l


---
## Section 5 -- Pretrained ResNet18 for Grayscale

In [ ]:
def build_grayscale_resnet18(emb_dim=256):
    try:
        from torchvision.models import ResNet18_Weights
        net = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    except ImportError:
        net = models.resnet18(pretrained=True)
    # Average RGB conv1 weights -> grayscale: [64,3,7,7] -> [64,1,7,7]
    w = net.conv1.weight.data
    gray_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    gray_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = gray_conv
    net.fc    = nn.Linear(net.fc.in_features, emb_dim)
    return net

backbone = build_grayscale_resnet18(EMB_DIM).to(DEVICE)
arc_head = ArcFaceLoss(EMB_DIM, NUM_CLASSES, scale=64.0, margin=0.5).to(DEVICE)

n_bb = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
n_hd = sum(p.numel() for p in arc_head.parameters()  if p.requires_grad)
print(f'Backbone params: {n_bb:,}  |  ArcFace params: {n_hd:,}')
with torch.no_grad():
    emb  = backbone(x_sample.to(DEVICE))
    loss = arc_head(emb, y_sample.to(DEVICE))
print(f'Initial loss   : {loss.item():.4f}  (expect ~log({NUM_CLASSES})={math.log(NUM_CLASSES):.2f} without scale)')


---
## Section 6 -- Phase 1: Full Network Training (30 epochs)

**Hard example mining**: after each epoch, per-class average loss is computed.
Mining weights update via EMA (alpha=0.3): classes that are harder to classify
receive higher sampling probability in the next epoch.

**LR schedule**: 3-epoch linear warmup (0.1x -> 1x) then cosine decay to 1e-5.

**Gradient clipping** (max_norm=1.0) prevents large gradient updates on pretrained weights.


In [ ]:
set_seed(SEED)

backbone = build_grayscale_resnet18(EMB_DIM).to(DEVICE)
arc_head = ArcFaceLoss(EMB_DIM, NUM_CLASSES, scale=64.0, margin=0.5).to(DEVICE)

params_p1    = list(backbone.parameters()) + list(arc_head.parameters())
optimizer_p1 = torch.optim.Adam(params_p1, lr=1e-3, weight_decay=1e-4)

N_P1 = 30
# 3-epoch warmup then cosine
warmup_s  = torch.optim.lr_scheduler.LinearLR(optimizer_p1, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine_s  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=N_P1-3, eta_min=1e-5)
sched_p1  = torch.optim.lr_scheduler.SequentialLR(optimizer_p1, schedulers=[warmup_s, cosine_s], milestones=[3])

mining_weights = torch.ones(NUM_CLASSES, dtype=torch.float32)
hist_p1        = {'loss':[], 'acc':[]}

for epoch in range(N_P1):
    loader_p1 = make_mining_loader(mining_weights)
    backbone.train(); arc_head.train()
    ep_loss  = 0.0; correct = 0; total = 0; n_batches = 0
    cls_loss_sum = torch.zeros(NUM_CLASSES)
    cls_loss_cnt = torch.zeros(NUM_CLASSES)

    for imgs, labels in tqdm(loader_p1, desc=f'P1 {epoch+1:02d}/{N_P1}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        emb      = backbone(imgs)
        per_loss = arc_head(emb, labels, reduction='none')  # [B] for mining
        loss     = per_loss.mean()

        optimizer_p1.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params_p1, max_norm=1.0)
        optimizer_p1.step()

        with torch.no_grad():
            lc = labels.cpu(); lv = per_loss.detach().cpu()
            cls_loss_sum.scatter_add_(0, lc, lv)
            cls_loss_cnt.scatter_add_(0, lc, torch.ones_like(lv))
            logits   = F.linear(F.normalize(emb,dim=1), F.normalize(arc_head.weight,dim=1)) * arc_head.scale
            correct += (logits.argmax(1)==labels.long()).sum().item()
            total   += labels.size(0)
        ep_loss  += loss.item(); n_batches += 1

    sched_p1.step()

    # EMA update: harder classes get higher weight
    avg_cls = cls_loss_sum / cls_loss_cnt.clamp(min=1)
    mining_weights = (0.7 * mining_weights + 0.3 * avg_cls).clamp(min=1e-6)

    avg = ep_loss/n_batches; acc = correct/total
    hist_p1['loss'].append(avg); hist_p1['acc'].append(acc)
    print(f'P1 E{epoch+1:02d}/{N_P1}  loss={avg:.4f}  acc={acc:.3f}  '
          f'lr={sched_p1.get_last_lr()[0]:.2e}  '
          f'hard_cls={mining_weights.argmax().item()} ({mining_weights.max().item():.2f}x)')


---
## Section 7 -- Phase 2: Fine-Tuning (15 epochs)

Freeze layers 1-3 of ResNet18; train only layer4 + fc + ArcFace head.
Hard mining continues with EMA weights inherited from Phase 1.
Lower LR: backbone=2e-4, ArcFace=5e-4.


In [ ]:
for name, p in backbone.named_parameters():
    p.requires_grad = not any(name.startswith(s) for s in ['conv1','bn1','layer1','layer2','layer3'])

n_frozen = sum(1 for p in backbone.parameters() if not p.requires_grad)
n_train  = sum(1 for p in backbone.parameters() if p.requires_grad)
print(f'Frozen: {n_frozen}  |  Trainable: {n_train}')

optimizer_p2 = torch.optim.Adam([
    {'params': [p for p in backbone.parameters() if p.requires_grad], 'lr': 2e-4},
    {'params': arc_head.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)

N_P2     = 15
sched_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=N_P2, eta_min=1e-6)

params_p2 = [p for p in backbone.parameters() if p.requires_grad] + list(arc_head.parameters())
hist_p2   = {'loss':[], 'acc':[]}

for epoch in range(N_P2):
    loader_p2 = make_mining_loader(mining_weights)
    backbone.train(); arc_head.train()
    ep_loss  = 0.0; correct = 0; total = 0; n_batches = 0
    cls_loss_sum = torch.zeros(NUM_CLASSES)
    cls_loss_cnt = torch.zeros(NUM_CLASSES)

    for imgs, labels in tqdm(loader_p2, desc=f'P2 {epoch+1:02d}/{N_P2}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        emb      = backbone(imgs)
        per_loss = arc_head(emb, labels, reduction='none')
        loss     = per_loss.mean()

        optimizer_p2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params_p2, max_norm=1.0)
        optimizer_p2.step()

        with torch.no_grad():
            lc = labels.cpu(); lv = per_loss.detach().cpu()
            cls_loss_sum.scatter_add_(0, lc, lv)
            cls_loss_cnt.scatter_add_(0, lc, torch.ones_like(lv))
            logits   = F.linear(F.normalize(emb,dim=1), F.normalize(arc_head.weight,dim=1)) * arc_head.scale
            correct += (logits.argmax(1)==labels.long()).sum().item()
            total   += labels.size(0)
        ep_loss  += loss.item(); n_batches += 1

    sched_p2.step()
    avg_cls = cls_loss_sum / cls_loss_cnt.clamp(min=1)
    mining_weights = (0.7 * mining_weights + 0.3 * avg_cls).clamp(min=1e-6)

    avg = ep_loss/n_batches; acc = correct/total
    hist_p2['loss'].append(avg); hist_p2['acc'].append(acc)
    print(f'P2 E{epoch+1:02d}/{N_P2}  loss={avg:.4f}  acc={acc:.3f}  lr={sched_p2.get_last_lr()[0]:.2e}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

e1 = range(1, N_P1+1)
e2 = range(N_P1+1, N_P1+N_P2+1)

axes[0].plot(e1, hist_p1['loss'], 'b-', lw=1.5, label='Phase 1')
axes[0].plot(e2, hist_p2['loss'], 'r-', lw=1.5, label='Phase 2')
axes[0].axvline(N_P1+0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('ArcFace Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(e1, hist_p1['acc'], 'b-', lw=1.5, label='Phase 1')
axes[1].plot(e2, hist_p2['acc'], 'r-', lw=1.5, label='Phase 2')
axes[1].axvline(N_P1+0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Train Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0,1); axes[1].legend(); axes[1].grid(True)

# Mining weight distribution after training
w_sorted = mining_weights.numpy().copy(); w_sorted.sort()
axes[2].bar(range(NUM_CLASSES), w_sorted[::-1], color='steelblue', alpha=0.7)
axes[2].set_title('Final Mining Weights (sorted, high=harder class)')
axes[2].set_xlabel('Class rank'); axes[2].set_ylabel('Weight'); axes[2].grid(True, alpha=0.4)

plt.tight_layout(); plt.show()


---
## Section 8 -- Evaluation Pairs (CEDAR test set)

In [ ]:
def build_pools(df, wid_set):
    gen, forg = {}, {}
    for wid, grp in df[df['writer_uid'].isin(wid_set)].groupby('writer_uid'):
        g = grp[grp['label']=='genuine']['path'].tolist()
        f = grp[grp['label']=='forgery']['path'].tolist()
        if g: gen[wid]  = g
        if f: forg[wid] = f
    return gen, forg

def generate_pairs(df, wid_set, n_pairs=10000, seed=1, neg_mix=0.8):
    gen, forg = build_pools(df, wid_set)
    writers   = sorted(gen); wf = sorted(set(gen)&set(forg))
    rng       = np.random.default_rng(seed)
    n_pos = n_pairs//2; n_neg = n_pairs-n_pos
    n_ns  = int(round(n_neg*neg_mix)); n_nc = n_neg-n_ns
    rows  = []
    for _ in range(n_pos):
        w=rng.choice(writers); g=gen[w]; i,j=rng.choice(len(g),size=2,replace=False)
        rows.append({'path_a':g[i],'path_b':g[j],'label':1,'pair_type':'pos'})
    for _ in range(n_ns):
        w=rng.choice(wf); g=gen[w]; f=forg[w]
        rows.append({'path_a':g[rng.integers(len(g))],'path_b':f[rng.integers(len(f))],'label':0,'pair_type':'neg_same'})
    for _ in range(n_nc):
        w1,w2=rng.choice(writers,size=2,replace=False)
        rows.append({'path_a':gen[w1][rng.integers(len(gen[w1]))],'path_b':gen[w2][rng.integers(len(gen[w2]))],'label':0,'pair_type':'neg_cross'})
    return pd.DataFrame(rows).sample(frac=1,random_state=seed).reset_index(drop=True)

def make_hard_pairs(df, wid_set, seed=5):
    gen, forg = build_pools(df, wid_set)
    rng = np.random.default_rng(seed); rows = []
    for w in sorted(set(gen)&set(forg)):
        g,f = gen[w],forg[w]
        for gi in g:
            for fi in f:
                rows.append({'path_a':gi,'path_b':fi,'label':0,'pair_type':'neg_same'})
        for _ in range(len(g)*len(f)):
            i,j=rng.choice(len(g),size=2,replace=False)
            rows.append({'path_a':g[i],'path_b':g[j],'label':1,'pair_type':'pos'})
    return pd.DataFrame(rows).sample(frac=1,random_state=seed).reset_index(drop=True)

test_pairs = generate_pairs(df_cedar, cedar_test_wids, n_pairs=10000, seed=3)
hard_pairs = make_hard_pairs(df_cedar, cedar_test_wids, seed=5)

test_ld = DataLoader(SiamesePairDataset(test_pairs, eval_tfm), batch_size=BATCH, shuffle=False, num_workers=0)
hard_ld = DataLoader(SiamesePairDataset(hard_pairs, eval_tfm), batch_size=BATCH, shuffle=False, num_workers=0)

print(f'Standard pairs : {len(test_pairs):,}')
print(f'Hard-neg pairs : {len(hard_pairs):,}')


---
## Section 9 -- Standard Metrics

In [ ]:
def extract_sims(loader, backbone, device):
    backbone.eval()
    sims, labs = [], []
    with torch.no_grad():
        for ia, ib, lbl in tqdm(loader, desc='Eval', leave=False):
            e1 = F.normalize(backbone(ia.to(device)), dim=1)
            e2 = F.normalize(backbone(ib.to(device)), dim=1)
            sims.append((e1*e2).sum(1).cpu().numpy())
            labs.append(lbl.numpy())
    return np.concatenate(sims), np.concatenate(labs).astype(int)

for p in backbone.parameters(): p.requires_grad_(False)

sim_std, y_std = extract_sims(test_ld, backbone, DEVICE)
res_std = compute_metrics(y_std, sim_std)

print(f'Cosine sim (pos): {sim_std[y_std==1].mean():.4f}')
print(f'Cosine sim (neg): {sim_std[y_std==0].mean():.4f}')
print(f'Gap             : {sim_std[y_std==1].mean()-sim_std[y_std==0].mean():.4f}')
print()
print('=== Standard Metrics ===')
for k,v in res_std.items(): print(f'  {k:<20s}: {v:.4f}')


In [ ]:
fpr,tpr,_ = sk_metrics.roc_curve(y_std, sim_std)
thr = np.linspace(sim_std.min(), sim_std.max(), 400)
fa,fr = [],[]
for t in thr:
    p=(sim_std>=t).astype(int)
    tp=int(((p==1)&(y_std==1)).sum()); fp=int(((p==1)&(y_std==0)).sum())
    fn=int(((p==0)&(y_std==1)).sum()); tn=int(((p==0)&(y_std==0)).sum())
    fa.append(fp/(fp+tn) if fp+tn>0 else 0.); fr.append(fn/(fn+tp) if fn+tp>0 else 0.)
fa=np.array(fa); fr=np.array(fr)

fig,axes = plt.subplots(1,3,figsize=(16,4))
axes[0].plot(fpr,tpr,'darkorange',lw=2,label=f"AUC={res_std['auc']:.4f}")
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC (Standard)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(thr,fa,'r-',label='FAR'); axes[1].plot(thr,fr,'g-',label='FRR')
axes[1].axvline(res_std['eer_threshold'],linestyle='--',color='gray',label=f"EER@{res_std['eer']:.4f}")
axes[1].set_title('FAR/FRR'); axes[1].set_xlabel('Threshold'); axes[1].legend(); axes[1].grid(True)
axes[2].hist(sim_std[y_std==1],bins=50,alpha=0.6,color='green',label='Positive')
axes[2].hist(sim_std[y_std==0],bins=50,alpha=0.6,color='red',  label='Negative')
axes[2].set_title('Similarity Distribution'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.show()


---
## Section 10 -- Hard-Negative Metrics (genuine vs skilled forgery)

In [ ]:
sim_hard, y_hard = extract_sims(hard_ld, backbone, DEVICE)
res_hard = compute_metrics(y_hard, sim_hard)

print(f'Hard -- Cosine sim (pos): {sim_hard[y_hard==1].mean():.4f}')
print(f'Hard -- Cosine sim (neg): {sim_hard[y_hard==0].mean():.4f}')
print(f'Hard -- Gap             : {sim_hard[y_hard==1].mean()-sim_hard[y_hard==0].mean():.4f}')
print()
print('=== Hard-Negative Metrics ===')
for k,v in res_hard.items(): print(f'  {k:<20s}: {v:.4f}')

fpr_h,tpr_h,_ = sk_metrics.roc_curve(y_hard, sim_hard)
fig,axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(fpr,   tpr,   'darkorange',lw=2,label=f"Standard AUC={res_std['auc']:.4f}")
axes[0].plot(fpr_h, tpr_h, 'navy',      lw=2,linestyle='--',label=f"Hard AUC={res_hard['auc']:.4f}")
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC: Standard vs Hard-Negative')
axes[0].legend(); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].grid(True)
axes[1].hist(sim_hard[y_hard==1],bins=50,alpha=0.6,color='green',label='Pos (genuine-genuine)')
axes[1].hist(sim_hard[y_hard==0],bins=50,alpha=0.6,color='red',  label='Hard neg (genuine-forgery)')
axes[1].set_title('Hard-Negative Similarity Distribution'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


---
## Section 11 -- Embedding Space (PCA)

In [ ]:
test_w2c   = {wid:i for i,wid in enumerate(sorted(cedar_test_wids))}
test_gen   = SignatureClassDataset(df_cedar, test_w2c, cedar_test_wids, transform=eval_tfm)
test_gen_l = DataLoader(test_gen, batch_size=BATCH, shuffle=False, num_workers=0)

backbone.eval()
all_e, all_w = [], []
with torch.no_grad():
    for imgs, cls in tqdm(test_gen_l, desc='PCA', leave=False):
        all_e.append(F.normalize(backbone(imgs.to(DEVICE)),dim=1).cpu().numpy())
        all_w.append(cls.numpy())
all_e = np.concatenate(all_e); all_w = np.concatenate(all_w)

pca   = PCA(n_components=2, random_state=SEED)
e2d   = pca.fit_transform(all_e)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}')

fig,ax = plt.subplots(figsize=(9,7))
cmap   = plt.cm.get_cmap('tab10', len(cedar_test_wids))
for i,wid in enumerate(sorted(test_w2c)):
    mask = all_w==test_w2c[wid]
    ax.scatter(e2d[mask,0], e2d[mask,1], color=cmap(i), label=wid, s=60, alpha=0.85)
ax.set_title('PCA -- CEDAR test writers (genuine) -- ArcFace v4')
ax.legend(loc='best', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## Section 12 -- Summary

In [ ]:
from IPython.display import Markdown, display

std_rows  = '\n'.join(f'| {k:<22s} | {v:.4f} |' for k,v in res_std.items())
hard_rows = '\n'.join(f'| {k:<22s} | {v:.4f} |' for k,v in res_hard.items())

gap_std  = sim_std[y_std==1].mean()  - sim_std[y_std==0].mean()
gap_hard = sim_hard[y_hard==1].mean()- sim_hard[y_hard==0].mean()

display(Markdown(f'''
### ArcFace v4 -- CEDAR test writers

#### Standard (5k pos + 5k neg, 80% same-writer forgeries)
| Metric                   | Value  |
|:-------------------------|-------:|
{std_rows}

#### Hard-negative (genuine vs skilled forgery only)
| Metric                   | Value  |
|:-------------------------|-------:|
{hard_rows}

### Version comparison
| Metric       | v1 SmallCNN 38cls | v2 ResNet18 38cls | v3 ResNet18-PT 251cls | v4 +mining 251cls |
|:-------------|:-----------------:|:-----------------:|:---------------------:|:-----------------:|
| AUC          | 0.8477 | 0.9320 | 0.9671 | {res_std["auc"]:.4f} |
| EER          | 0.2584 | 0.1374 | 0.0889 | {res_std["eer"]:.4f} |
| EER (hard)   | --     | --     | 0.0910 | {res_hard["eer"]:.4f} |
| Cosine gap   | 0.0196 | 0.4434 | 0.4415 | {gap_std:.4f} |
| EER threshold| 0.9972 | 0.7131 | 0.5162 | {res_std["eer_threshold"]:.4f} |
'''))
